# 06 — Đánh giá mô hình

Đọc kết quả đã lưu ở notebook 05, kiểm tra lại điểm và xem mô hình sai ở đâu. Không huấn luyện lại và không tạo submission.

Đặt file trong `notebooks/`, chạy từ trên xuống. Cần chạy xong 05 và giữ thư mục dự báo validation. Notebook đọc theo từng batch để giảm RAM.

Dùng các thư viện đã có ở bước trước: numpy, pandas, pyarrow, matplotlib. Các biểu đồ và bảng được lưu trong `reports/06_runs/<run_id>/`.


## 1. Chọn lần chạy

Để `RUN_ID = None` nếu muốn dùng lần chạy hoàn tất gần nhất của 05. Điền mã lần chạy nếu cần xem lại kết quả cũ. Không trộn kết quả từ nhiều lần chạy.


In [ ]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import json, hashlib, sqlite3, tempfile
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

RUN_ID = None
BATCH_SIZE = 32768
SAMPLE_SIZE = 4000
SEED = 42
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'reports/03_config.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Mở notebook trong thư mục dự án có reports/03_config.json.')
if RUN_ID is None:
    latest = json.loads((ROOT / 'reports/05_latest_run.json').read_text(encoding='utf-8'))
    RUN_ID = latest['run_id']
RUN_DIR = ROOT / 'reports/05_runs' / RUN_ID
PRED_DIR = ROOT / 'data/interim/05_runs' / RUN_ID
OUT = ROOT / 'reports/06_runs' / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
report05 = json.loads((RUN_DIR / '05_model_training_report.json').read_text(encoding='utf-8'))
assert report05['run_id'] == RUN_ID
print('Lần chạy:', RUN_ID)
print('Kết quả lưu tại:', OUT)


## 2. Kiểm tra nguồn dữ liệu

So khớp cấu hình và danh sách file với bước 05. Kiểm tra này không thay thế việc kiểm tra nội dung từng file Parquet. Nếu đã tạo lại feature, cần chạy lại 04 và 05 trước.


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

config_path = ROOT / 'reports/03_config.json'
manifest_path = ROOT / 'reports/table/03_feature_manifest.csv'
for path, key in [(config_path, 'source_config_sha256'),
                  (manifest_path, 'source_manifest_sha256')]:
    assert sha256(path) == report05[key], f'{path.name} khác lần chạy 05.'
config = json.loads(config_path.read_text(encoding='utf-8'))
manifest = pd.read_csv(manifest_path)
manifest['origin'] = pd.to_datetime(manifest.origin)
folds = config['folds']
assert len(folds) > 0
scores05 = pd.read_csv(RUN_DIR / 'scores.csv', dtype={'value': str})
assert not scores05.duplicated(['fold','experiment','predictor','group','value']).any()
assert np.isfinite(scores05.NWRMSLE).all()
print('Cấu hình và danh sách file khớp với 05.')


## 3. Cách tính điểm

NWRMSLE càng thấp càng tốt. Điểm được tính trên `log1p(doanh số)`, với trọng số 1,25 cho hàng dễ hỏng và 1 cho hàng còn lại.

Cộng tổng sai số bình phương có trọng số rồi mới lấy căn. Không lấy trung bình điểm của các batch. Baseline ở đây là `B2c_mean28`, đúng với bước 05.

Dự báo đã lưu dạng float32 nên điểm tính lại có thể lệch rất nhỏ so với 05.


In [ ]:
def add_stats(acc, name, group, value, y, p, w):
    e = p - y
    z = acc[(name, group, str(value))]
    z += np.array([len(y), w.sum(), np.dot(w, e * e), np.dot(w, e)], dtype='float64')

def evaluate_file(path, fold):
    cols = ['store_nbr','item_nbr','origin','date','h','pair_status',
            'perishable','item_family','y_log','w','pred_log','baseline_log']
    pf = pq.ParquetFile(path)
    assert set(cols).issubset(pf.schema_arrow.names), f'Thiếu cột: {path.name}'
    acc = defaultdict(lambda: np.zeros(4, dtype='float64'))
    rng = np.random.default_rng(SEED)
    sample = pd.DataFrame()
    cut = pd.Timestamp(fold['valid_origin'])
    total = 0
    # Dùng ổ đĩa để kiểm tra khóa trùng giữa các batch, tránh giữ hết khóa trong RAM.
    with tempfile.TemporaryDirectory(prefix='eval06_') as tmp:
        db = sqlite3.connect(str(Path(tmp) / 'keys.sqlite'))
        try:
            db.execute('CREATE TABLE keys (s INTEGER, i INTEGER, d TEXT, PRIMARY KEY(s,i,d)) WITHOUT ROWID')
            for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=cols):
                d = batch.to_pandas()
                assert not d[cols].isna().any().any(), f'Có giá trị thiếu: {path.name}'
                d['origin'] = pd.to_datetime(d.origin)
                d['date'] = pd.to_datetime(d.date)
                assert (d.origin == cut).all()
                assert ((d.date - d.origin).dt.days == d.h).all()
                assert d.h.between(1, int(config['horizon'])).all()
                assert d.pair_status.isin([0,1,2,3]).all()
                assert d.perishable.isin([0,1]).all()
                for col in ['store_nbr','item_nbr','h','pair_status']:
                    assert np.isfinite(d[col]).all() and (d[col] == np.floor(d[col])).all()
                keys = zip(d.store_nbr.astype(int), d.item_nbr.astype(int), d.date.astype(str))
                try:
                    db.executemany('INSERT INTO keys VALUES (?,?,?)', keys)
                except sqlite3.IntegrityError as exc:
                    raise AssertionError(f'Trùng khóa cửa hàng, mặt hàng, ngày: {path.name}') from exc
                db.commit()
                y = d.y_log.to_numpy(dtype='float64')
                w = d.w.to_numpy(dtype='float64')
                assert np.isfinite(y).all() and (y >= 0).all()
                assert np.isfinite(w).all() and (w > 0).all()
                assert np.allclose(w, np.where(d.perishable == 1, 1.25, 1.0))
                for name, col in [('model','pred_log'), ('B2c_mean28','baseline_log')]:
                    p = d[col].to_numpy(dtype='float64')
                    assert np.isfinite(p).all() and (p >= 0).all()
                    add_stats(acc, name, 'overall', 'all', y, p, w)
                    for group in ['h','pair_status','perishable','item_family']:
                        for value, index in d.groupby(group, observed=True).indices.items():
                            add_stats(acc, name, group, int(value), y[index], p[index], w[index])
                d['_random'] = rng.random(len(d))
                sample = pd.concat([sample, d], ignore_index=True).nsmallest(SAMPLE_SIZE, '_random')
                total += len(d)
        finally:
            db.close()
    expected = manifest[(manifest.role == fold['valid_role']) & (manifest.origin == cut)]
    assert len(expected) == 1 and total == int(expected.rows.iloc[0]) and total > 0
    rows = []
    for (predictor, group, value), (n, sw, sse, se) in acc.items():
        rows.append(dict(predictor=predictor, group=group, value=value, rows=int(n),
                         weight_sum=sw, weighted_sse=sse, NWRMSLE=np.sqrt(sse/sw), bias_log=se/sw))
    return pd.DataFrame(rows), sample.drop(columns='_random')


## 4. Tính lại điểm từ dự báo

Bắt buộc có dự báo `safe` cho từng fold. Nếu 05 đã lưu dự báo Ridge hoặc Extra Trees thì chúng cũng được đánh giá. Cấu hình chỉ có bảng điểm mà chưa lưu dự báo vẫn xuất hiện ở bảng so sánh, nhưng chưa được tính lại điểm tại đây.


In [ ]:
evaluated = []
samples = {}
checks = []
prediction_hashes = {}
for fold in folds:
    fold_name = fold['fold']
    experiments = scores05.loc[scores05.fold == fold_name, 'experiment'].unique()
    assert 'safe' in experiments, f'Thiếu safe ở {fold_name}'
    for exp in experiments:
        path = PRED_DIR / f'{fold_name}_{exp}.parquet'
        if not path.exists():
            assert exp != 'safe', f'Thiếu {path}. Chạy 05 với SAVE_SAFE_VALID_PREDICTIONS=True.'
            print('Chưa có file dự báo:', fold_name, exp)
            continue
        table, sample = evaluate_file(path, fold)
        table['fold'] = fold_name
        table['experiment'] = exp
        ref = scores05[(scores05.fold == fold_name) & (scores05.experiment == exp)]
        merged = table.merge(ref, on=['predictor','group','value'], suffixes=('_06','_05'), validate='one_to_one')
        assert len(merged) == len(table) == len(ref), 'Nhóm đánh giá khác 05.'
        assert np.array_equal(merged.rows_06, merged.rows_05)
        assert np.allclose(merged.weight_sum_06, merged.weight_sum_05, rtol=1e-8, atol=1e-6)
        assert np.allclose(merged.NWRMSLE_06, merged.NWRMSLE_05, rtol=1e-5, atol=1e-6), 'Điểm khác 05.'
        checks.append(dict(fold=fold_name, experiment=exp, status='OK',
                           max_score_difference=float(abs(merged.NWRMSLE_06-merged.NWRMSLE_05).max())))
        prediction_hashes[path.name] = sha256(path)
        evaluated.append(table)
        samples[(fold_name, exp)] = sample
metrics = pd.concat(evaluated, ignore_index=True)
checks = pd.DataFrame(checks)
metrics.to_csv(OUT / '06_metrics.csv', index=False)
checks.to_csv(OUT / '06_validation_checks.csv', index=False)
display(checks)


## 5. So sánh với baseline

Mức cải thiện dương nghĩa là mô hình có điểm thấp hơn baseline. Xem riêng từng fold để tránh che mất giai đoạn mô hình hoạt động kém. Các biến thể khuyến mãi hoặc dầu của LightGBM không phải là thuật toán khác.


In [ ]:
comparison = scores05[scores05.group == 'overall'].pivot(
    index=['fold','experiment'], columns='predictor', values='NWRMSLE').reset_index()
comparison['improvement_pct'] = 100 * (comparison.B2c_mean28 - comparison.model) / comparison.B2c_mean28
verified = set(zip(checks.fold, checks.experiment))
comparison['recomputed_06'] = [(f,e) in verified for f,e in zip(comparison.fold, comparison.experiment)]
comparison.to_csv(OUT / '06_comparison.csv', index=False)
display(comparison.round(6))
fig, axes = plt.subplots(1, len(folds), figsize=(7*len(folds), 4), squeeze=False)
for ax, fold in zip(axes[0], folds):
    d = comparison[comparison.fold == fold['fold']].set_index('experiment')
    d[['model','B2c_mean28']].plot.bar(ax=ax, rot=25)
    ax.set(title=fold['fold'], ylabel='NWRMSLE — thấp hơn tốt hơn', xlabel='')
    ax.grid(axis='y', alpha=.2)
fig.tight_layout()
fig.savefig(OUT / '06_model_comparison.png', dpi=150)
plt.show()


## 6. Sai số theo ngày dự báo

`h = 1` là ngày sau mốc dự báo; `h = 16` là ngày xa nhất. Đường mô hình nằm trên baseline nghĩa là mô hình kém hơn ở ngày đó.


In [ ]:
fig, axes = plt.subplots(1, len(folds), figsize=(7*len(folds), 4), squeeze=False)
for ax, fold in zip(axes[0], folds):
    d = metrics[(metrics.fold == fold['fold']) & (metrics.group == 'h')].copy()
    d['h'] = d.value.astype(int)
    for (exp, predictor), g in d.groupby(['experiment','predictor']):
        if predictor == 'B2c_mean28' and exp != 'safe':
            continue
        g = g.sort_values('h')
        ax.plot(g.h, g.NWRMSLE, marker='.', label=exp if predictor == 'model' else predictor)
    ax.set(title=fold['fold'], xlabel='Ngày dự báo (h)', ylabel='NWRMSLE')
    ax.legend(); ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(OUT / '06_horizon.png', dpi=150)
plt.show()


## 7. Nhóm nào còn dự báo kém?

Nhóm 0 có bán trong 28 ngày gần nhất. Nhóm 1 có lịch sử nhưng không có bán dương trong cửa sổ đó. Nhóm 2 là cặp cửa hàng–mặt hàng mới; nhóm 3 là mặt hàng mới.

`bias_log` dương: thường dự báo cao; âm: thường dự báo thấp. Đây là chênh lệch trên thang log, không phải số lượng sản phẩm.

Bảng nhóm hàng giữ mã `item_family` như bước 03. Nhóm có tổng sai số lớn đáng ưu tiên xem xét, nhưng không nhất thiết có điểm trung bình cao nhất.


In [ ]:
safe = metrics[metrics.experiment == 'safe']
for group in ['pair_status','perishable']:
    display(safe[safe.group == group][['fold','predictor','value','rows','weight_sum','NWRMSLE','bias_log']].round(6))
family = safe[(safe.group == 'item_family') & (safe.predictor == 'model')].copy()
family['error_share'] = family.weighted_sse / family.groupby('fold').weighted_sse.transform('sum').replace(0, np.nan)
for fold in folds:
    print(fold['fold'])
    display(family[family.fold == fold['fold']].nlargest(10, 'weighted_sse')[
        ['value','rows','NWRMSLE','bias_log','error_share']].round(6))
family.to_csv(OUT / '06_family_errors.csv', index=False)


## 8. Giá trị thực tế và dự báo

Biểu đồ dùng mẫu ngẫu nhiên tối đa 4.000 dòng mỗi file; điểm số ở trên dùng toàn bộ validation. Không dùng mẫu biểu đồ để tính điểm thay thế.

Ở hình trái, điểm càng sát đường chéo càng tốt. Ở hình phải, sai số bằng dự báo trừ thực tế; đường 0 là dự báo đúng. Cả hai hình dùng thang `log1p` để nhìn rõ cả doanh số nhỏ và lớn.


In [ ]:
for (fold_name, exp), d in samples.items():
    fig, axes = plt.subplots(1, 2, figsize=(11,4))
    top = max(float(d.y_log.max()), float(d.pred_log.max()), 1.)
    axes[0].scatter(d.y_log, d.pred_log, s=7, alpha=.2)
    axes[0].plot([0,top], [0,top], 'r--', linewidth=1)
    axes[0].set(xlabel='Thực tế (log1p)', ylabel='Dự báo (log1p)', xlim=(0,top), ylim=(0,top))
    axes[1].scatter(d.pred_log, d.pred_log-d.y_log, s=7, alpha=.2)
    axes[1].axhline(0, color='r', linestyle='--', linewidth=1)
    axes[1].set(xlabel='Dự báo (log1p)', ylabel='Sai số (log1p)')
    fig.suptitle(f'{fold_name} — {exp} — mẫu {len(d):,} dòng')
    fig.tight_layout()
    fig.savefig(OUT / f'06_scatter_residual_{fold_name}_{exp}.png', dpi=150)
    plt.show()


## 9. Kết quả này áp dụng đến đâu?

Validation có thể chỉ chứa nhóm đã có lịch sử. Khi đó chưa biết cách dự báo dự phòng cho nhóm mới tốt hay kém. Không suy điểm của toàn bộ test từ điểm của nhóm đã có lịch sử.

V1 và V2 đã dùng để so sánh mô hình, nên không còn là tập kiểm tra hoàn toàn độc lập. Điểm tốt hơn trên hai fold chưa chứng minh mọi giai đoạn đều tốt hơn.


In [ ]:
coverage = safe[(safe.group == 'pair_status') & (safe.predictor == 'model')].copy()
coverage['weight_share'] = coverage.weight_sum / coverage.groupby('fold').weight_sum.transform('sum')
display(coverage[['fold','value','rows','weight_share','NWRMSLE']].round(6))
coverage.to_csv(OUT / '06_validation_coverage.csv', index=False)
final_info = report05.get('final_model', {})
test_counts = final_info.get('test_rows_by_status', {})
if test_counts:
    test_coverage = pd.DataFrame([{'pair_status':int(k), 'rows':int(v)} for k,v in test_counts.items()])
    test_coverage['row_share'] = test_coverage.rows / test_coverage.rows.sum()
    display(test_coverage.sort_values('pair_status'))
    test_coverage.to_csv(OUT / '06_test_row_coverage.csv', index=False)
    print('Tỷ lệ trên là tỷ lệ dòng test, không phải tỷ lệ trọng số hay điểm test.')
else:
    print('05 chưa lưu số dòng test theo nhóm; chưa tính được độ phủ test tại đây.')
print('Dự báo test hữu hạn chỉ xác nhận chạy được, không xác nhận dự báo đúng.')


## 10. Nhận xét và bước tiếp theo

Phần dưới tự ghi nhận xét từ kết quả vừa kiểm tra. Nếu kiểm tra dữ liệu hoặc điểm số lỗi, dừng và sửa trước khi viết kết luận.

Nếu mô hình tốt hơn baseline ở mọi fold, có thể viết 07 để tổng hợp kết quả hiện có và nêu rõ giới hạn. Nếu muốn cải thiện mô hình, quay lại 03 hoặc 05, rồi đánh giá lại. Không chọn cách ghép mô hình theo nhóm chỉ vì nó tốt hơn trên chính hai fold đã xem.


In [ ]:
safe_comparison = comparison[comparison.experiment == 'safe']
assert len(safe_comparison) == len(folds)
better_all = bool((safe_comparison.improvement_pct > 0).all())
lines = [f'# Đánh giá mô hình — {RUN_ID}', '',
         'Đã kiểm tra khóa trùng, số dòng, ngày dự báo, trọng số và điểm tính lại.', '']
for row in safe_comparison.itertuples():
    lines.append(f'- {row.fold}: safe {row.model:.6f}; baseline {row.B2c_mean28:.6f}; cải thiện {row.improvement_pct:.2f}%.')
lines += ['', 'Mô hình safe tốt hơn baseline trên mọi fold.' if better_all else
          'Mô hình safe chưa tốt hơn baseline trên mọi fold. Cần xem lại 05 trước khi chốt mô hình.']
status_scores = safe[safe.group == 'pair_status'].pivot(index=['fold','value'], columns='predictor', values='NWRMSLE')
for (fold_name, value), row in status_scores.iterrows():
    if row['model'] > row['B2c_mean28']:
        lines.append(f'- {fold_name}, nhóm {value}: mô hình kém baseline; cần xem kỹ nhóm này.')
missing_cold = [f['fold'] for f in folds if not {'2','3'}.issubset(
    set(coverage.loc[coverage.fold == f['fold'], 'value']))]
if missing_cold:
    lines += ['', 'Chưa đánh giá đủ nhóm cặp mới và mặt hàng mới ở: ' + ', '.join(missing_cold) + '.']
lines += ['', 'Chưa có điểm test. Hai fold đã dùng để so sánh mô hình; không phải kiểm tra độc lập.',
          'Chưa thử đầy đủ cách xử lý ngày không có dòng. Khuyến mãi tương lai vẫn cần kiểm chứng.', '',
          'Có thể viết 07 để tổng hợp kết quả và các giới hạn trên.' if better_all else
          'Ưu tiên kiểm tra nhóm bị giảm điểm và cấu hình ở 05, sau đó chạy lại 06.']
text = '\n'.join(lines) + '\n'
(OUT / '06_evaluation_summary.md').write_text(text, encoding='utf-8')
report06 = dict(run_id=RUN_ID, created_at_utc=datetime.now(timezone.utc).isoformat(),
    source_report_sha256=sha256(RUN_DIR / '05_model_training_report.json'),
    prediction_sha256=prediction_hashes, checks=checks.to_dict('records'),
    safe_beats_baseline_all_folds=better_all, folds_without_full_cold_start=missing_cold,
    independent_holdout=False, test_accuracy_known=False,
    comparison=comparison.to_dict('records'))
(OUT / '06_model_evaluation_report.json').write_text(
    json.dumps(report06, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
display(Markdown(text))
print('Đã lưu bảng, hình và báo cáo:', OUT)
